# Step 1: Install dependencies

In [1]:
# Install PySpark and Delta for Colab (skip in Databricks)
!pip install pyspark==3.5.1 delta-spark==3.1.0 -q
!pip install delta-spark==3.2.0 -q

import pandas as pd
import numpy as np
import pyspark
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql.functions import col, datediff, current_date, when

# Step 2: Upload datasets

In [2]:
from google.colab import files

print("Upload customers.csv")
customers_file = files.upload()
customers_file = list(customers_file.keys())[0]

print("Upload orders.csv")
orders_file = files.upload()
orders_file = list(orders_file.keys())[0]

print("Upload delivery_status.csv")
delivery_file = files.upload()
delivery_file = list(delivery_file.keys())[0]

print("Upload cleaned_orders.csv")
cleaned_orders_file = files.upload()
cleaned_orders_file = list(cleaned_orders_file.keys())[0]


Upload customers.csv


Saving customers.csv to customers.csv
Upload orders.csv


Saving orders.csv to orders.csv
Upload delivery_status.csv


Saving delivery_status.csv to delivery_status.csv
Upload cleaned_orders.csv


Saving cleaned_orders.csv to cleaned_orders.csv


# Step 3: Load datasets with Pandas

In [3]:
customers_df = pd.read_csv(customers_file)
orders_df = pd.read_csv(orders_file, parse_dates=['order_date','delivery_date'])
delivery_status_df = pd.read_csv(delivery_file, parse_dates=['last_updated'])

print("Customers Data:")
print(customers_df.head())

print("\nOrders Data:")
print(orders_df.head())

print("\nDelivery Status Data:")
print(delivery_status_df.head())


Customers Data:
   customer_id          name               email       phone   region
0            1   Rahul Kumar   rahul@example.com  9876543210    North
1            2  Anita Sharma   anita@example.com  9123456780    South
2            3     Vijay Rao   vijay@example.com  9988776655     East
3            4  Swathi Menon  swathi@example.com  9090909090     West
4            5     Arjun Das   arjun@example.com  9012345678  Central

Orders Data:
   order_id  customer_id order_date delivery_date     status
0         1            1 2025-07-01    2025-07-03  Delivered
1         2            2 2025-07-05    2025-07-08  Delivered
2         3            3 2025-07-10    2025-07-12    Pending
3         4            4 2025-07-12    2025-07-15    Shipped
4         5            1 2025-07-18    2025-07-20    Pending

Delivery Status Data:
   delivery_id  order_id current_status        last_updated
0            1         1      Delivered 2025-07-03 10:00:00
1            2         2      Delivered 2

# Step 4: Delay calculation

In [4]:
# Merge orders with delivery status
merged_df = pd.merge(orders_df, delivery_status_df, on='order_id', how='inner')

# Calculate delay_days
merged_df['delay_days'] = (merged_df['last_updated'] - merged_df['delivery_date']).dt.days
merged_df['delay_days'] = merged_df['delay_days'].apply(lambda x: x if x > 0 else 0)

# Delayed flag
merged_df['delayed'] = np.where(merged_df['delay_days'] > 0, 1, 0)

print("Merged & Delay Data:")
print(merged_df)


Merged & Delay Data:
   order_id  customer_id order_date delivery_date     status  delivery_id  \
0         1            1 2025-07-01    2025-07-03  Delivered            1   
1         2            2 2025-07-05    2025-07-08  Delivered            2   
2         3            3 2025-07-10    2025-07-12    Pending            3   
3         4            4 2025-07-12    2025-07-15    Shipped            4   
4         5            1 2025-07-18    2025-07-20    Pending            5   
5         6            5 2025-07-20    2025-07-23  Delivered            6   
6         7            6 2025-07-22    2025-07-25    Shipped            7   

  current_status        last_updated  delay_days  delayed  
0      Delivered 2025-07-03 10:00:00           0        0  
1      Delivered 2025-07-08 15:30:00           0        0  
2     In Transit 2025-07-20 11:45:00           8        1  
3        Shipped 2025-07-21 09:00:00           6        1  
4        Pending 2025-07-23 08:00:00           3        1  
5 

# Step 5: Top delayed customers and Most common delivery issues

In [5]:
final_df = pd.merge(merged_df, customers_df, on='customer_id', how='inner')

delay_summary = final_df.groupby(['customer_id','name'])['delayed'].sum().reset_index()
delay_summary = delay_summary.sort_values(by='delayed', ascending=False)

print("Top Delayed Customers:")
print(delay_summary)

issue_summary = merged_df['current_status'].value_counts()

print("Most Common Delivery Issues:")
print(issue_summary)



Top Delayed Customers:
   customer_id          name  delayed
0            1   Rahul Kumar        1
2            3     Vijay Rao        1
3            4  Swathi Menon        1
1            2  Anita Sharma        0
4            5     Arjun Das        0
5            6    Pooja Nair        0
Most Common Delivery Issues:
current_status
Delivered     3
Shipped       2
In Transit    1
Pending       1
Name: count, dtype: int64


# Step 6: Start Spark with Delta and load data

In [6]:
builder = pyspark.sql.SparkSession.builder.appName("CustomerOrderDelta") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("Spark with Delta is ready.")

spark_orders = spark.createDataFrame(orders_df)
spark_customers = spark.createDataFrame(customers_df)
spark_status = spark.createDataFrame(delivery_status_df)

spark_orders.show()
spark_customers.show()
spark_status.show()



Spark with Delta is ready.
+--------+-----------+-------------------+-------------------+---------+
|order_id|customer_id|         order_date|      delivery_date|   status|
+--------+-----------+-------------------+-------------------+---------+
|       1|          1|2025-07-01 00:00:00|2025-07-03 00:00:00|Delivered|
|       2|          2|2025-07-05 00:00:00|2025-07-08 00:00:00|Delivered|
|       3|          3|2025-07-10 00:00:00|2025-07-12 00:00:00|  Pending|
|       4|          4|2025-07-12 00:00:00|2025-07-15 00:00:00|  Shipped|
|       5|          1|2025-07-18 00:00:00|2025-07-20 00:00:00|  Pending|
|       6|          5|2025-07-20 00:00:00|2025-07-23 00:00:00|Delivered|
|       7|          6|2025-07-22 00:00:00|2025-07-25 00:00:00|  Shipped|
+--------+-----------+-------------------+-------------------+---------+

+-----------+------------+------------------+----------+-------+
|customer_id|        name|             email|     phone| region|
+-----------+------------+-------------

# Step 7: Delay calculation in PySpark

In [7]:
# Calculate delays with current_date
spark_orders = spark_orders.withColumn(
    "delay_days",
    datediff(current_date(), col("delivery_date"))
).withColumn(
    "delayed",
    when(col("delay_days") > 0, 1).otherwise(0)
)

spark_orders.show()


+--------+-----------+-------------------+-------------------+---------+----------+-------+
|order_id|customer_id|         order_date|      delivery_date|   status|delay_days|delayed|
+--------+-----------+-------------------+-------------------+---------+----------+-------+
|       1|          1|2025-07-01 00:00:00|2025-07-03 00:00:00|Delivered|        56|      1|
|       2|          2|2025-07-05 00:00:00|2025-07-08 00:00:00|Delivered|        51|      1|
|       3|          3|2025-07-10 00:00:00|2025-07-12 00:00:00|  Pending|        47|      1|
|       4|          4|2025-07-12 00:00:00|2025-07-15 00:00:00|  Shipped|        44|      1|
|       5|          1|2025-07-18 00:00:00|2025-07-20 00:00:00|  Pending|        39|      1|
|       6|          5|2025-07-20 00:00:00|2025-07-23 00:00:00|Delivered|        36|      1|
|       7|          6|2025-07-22 00:00:00|2025-07-25 00:00:00|  Shipped|        34|      1|
+--------+-----------+-------------------+-------------------+---------+--------

# Step 8: Join Orders + Customers, Group by region

In [8]:
orders_customers = spark_orders.join(spark_customers, on="customer_id", how="left")

region_summary = orders_customers.groupBy("region") \
    .agg(F.sum("delayed").alias("total_delayed_orders")) \
    .orderBy(col("total_delayed_orders").desc())

region_summary.show()

output_path = "/content/delayed_orders_by_region"
region_summary.coalesce(1).write.mode("overwrite").option("header","true").csv(output_path)

print("Output saved to CSV:", output_path)



+-------+--------------------+
| region|total_delayed_orders|
+-------+--------------------+
|  South|                   2|
|  North|                   2|
|Central|                   1|
|   East|                   1|
|   West|                   1|
+-------+--------------------+

Output saved to CSV: /content/delayed_orders_by_region


# Step 9: Pipeline update with delivery status

In [9]:
# Join orders with status
orders_updated_df = (
    spark_orders.alias("orders")
    .join(spark_status.alias("status"), on="order_id", how="left")
    .select(
        "orders.*",
        F.col("status.current_status").alias("latest_status"),
        F.col("status.last_updated").alias("status_update_time")
    )
)

# Calculate delay against last_updated
orders_updated_df = orders_updated_df.withColumn(
    "delay_days",
    F.when(
        (F.col("delivery_date").isNotNull()) & (F.col("status_update_time").isNotNull()),
        F.datediff(F.to_date(F.col("status_update_time")), F.to_date(F.col("delivery_date")))
    ).otherwise(0)
).withColumn(
    "delayed", when(col("delay_days") > 0, 1).otherwise(0)
)

orders_updated_df.show()


+--------+-----------+-------------------+-------------------+---------+----------+-------+-------------+-------------------+
|order_id|customer_id|         order_date|      delivery_date|   status|delay_days|delayed|latest_status| status_update_time|
+--------+-----------+-------------------+-------------------+---------+----------+-------+-------------+-------------------+
|       1|          1|2025-07-01 00:00:00|2025-07-03 00:00:00|Delivered|         0|      0|    Delivered|2025-07-03 10:00:00|
|       3|          3|2025-07-10 00:00:00|2025-07-12 00:00:00|  Pending|         8|      1|   In Transit|2025-07-20 11:45:00|
|       2|          2|2025-07-05 00:00:00|2025-07-08 00:00:00|Delivered|         0|      0|    Delivered|2025-07-08 15:30:00|
|       7|          6|2025-07-22 00:00:00|2025-07-25 00:00:00|  Shipped|         0|      0|      Shipped|2025-07-25 16:10:00|
|       6|          5|2025-07-20 00:00:00|2025-07-23 00:00:00|Delivered|         0|      0|    Delivered|2025-07-23 14

# Step 10  :Save results as Delta and CSV

In [10]:
# Save as Delta
orders_updated_df.write.format("delta").mode("overwrite").save("/content/orders_with_status_delta")

# Save as CSV
orders_updated_df.write.mode("overwrite").option("header","true").csv("/content/orders_with_status_csv")

print("Saved both Delta and CSV outputs")


Saved both Delta and CSV outputs
